In [1]:
from __future__ import annotations

from dataclasses import dataclass

from pydantic import BaseModel

from data_repository.electricity_mix_repository import electricity_mixes
from data_repository.model_repository import ParametersMoE, models
from data_repository.provider_repository import ProviderConfig, providers

from utils import llm_infer_impacts, llm_train_impacts, llm_train_data_storage_impacts
import datetime

In [2]:
models.find_model("openai", "gpt-4o-mini") 

Model(provider=<Providers.openai: 'openai'>, name='gpt-4o-mini', architecture=Architecture(type=<ArchitectureTypes.DENSE: 'dense'>, parameters=RangeValue(min=8, max=28)), warnings=[ModelArchNotReleasedWarning(code='model-arch-not-released', message='The model architecture has not been released, expect lower precision.'), ModelArchMultimodalWarning(code='model-arch-multimodal', message='The model architecture is multimodal, expect lower precision.')], sources=['https://platform.openai.com/docs/models#gpt-4o-mini'], deployment=Deployment(tps=34.5, ttft=0.51), publication_date=datetime.datetime(2025, 1, 1, 0, 0))

In [3]:
print(electricity_mixes.find_electricity_mix("ARG"))


ElectricityMix(zone='ARG', adpe=7.686e-08, pe=7.8888, gwp=0.35895, wue=11.4558)


In [4]:

models.list_models()

[Model(provider=<Providers.cohere: 'cohere'>, name='c4ai-aya-expanse-8b', architecture=Architecture(type=<ArchitectureTypes.DENSE: 'dense'>, parameters=8.03), warnings=[], sources=['https://docs.cohere.com/docs/models', 'https://huggingface.co/CohereLabs/aya-expanse-8b'], deployment=Deployment(tps=23.1, ttft=0.56), publication_date=datetime.datetime(2025, 1, 1, 0, 0)),
 Model(provider=<Providers.cohere: 'cohere'>, name='c4ai-aya-expanse-32b', architecture=Architecture(type=<ArchitectureTypes.DENSE: 'dense'>, parameters=32.3), warnings=[], sources=['https://docs.cohere.com/docs/models', 'https://huggingface.co/CohereLabs/aya-expanse-32b'], deployment=Deployment(tps=23.1, ttft=0.56), publication_date=datetime.datetime(2025, 1, 1, 0, 0)),
 Model(provider=<Providers.cohere: 'cohere'>, name='c4ai-aya-vision-8b', architecture=Architecture(type=<ArchitectureTypes.DENSE: 'dense'>, parameters=8.63), warnings=[ModelArchMultimodalWarning(code='model-arch-multimodal', message='The model architectu

In [5]:
providers.list_providers()

[ProviderConfig(name='anthropic', datacenter_location='USA', datacenter_pue=RangeValue(min=1.09, max=1.14), datacenter_wue=RangeValue(min=0.13, max=0.999), compute_capacity={'2020': None, '2021': None, '2022': None, '2023': None, '2024': None, '2025': 1.4, '2026': None}, number_of_active_models={'2020': None, '2021': None, '2022': None, '2023': None, '2024': None, '2025': 6, '2026': None}),
 ProviderConfig(name='cohere', datacenter_location='USA', datacenter_pue=1.09, datacenter_wue=0.999, compute_capacity={'2020': None, '2021': None, '2022': None, '2023': None, '2024': None, '2025': 1.9, '2026': None}, number_of_active_models={'2020': None, '2021': None, '2022': None, '2023': None, '2024': None, '2025': 10, '2026': None}),
 ProviderConfig(name='google_genai', datacenter_location='USA', datacenter_pue=1.09, datacenter_wue=0.999, compute_capacity={'2020': None, '2021': None, '2022': None, '2023': None, '2024': None, '2025': 3.0, '2026': None}, number_of_active_models={'2020': None, '202

In [6]:
# test inference
test_cases = [
    {'provider': 'openai', 'model': 'gpt-4o-mini', 'tokens': 400},
    {'provider': 'anthropic', 'model': 'claude-haiku-4-5', 'tokens': 400},
    {'provider': 'openai', 'model': 'gpt-4o', 'tokens': 400},
]

print('='*60)
print('LLM Impacts test')
print('='*60)

for case in test_cases:
    result = llm_infer_impacts(
        provider=case['provider'],
        model_name=case['model'],
        output_token_count=case['tokens'],
        electricity_mix_zone='USA'
    )
    
    print(f"\nModel name: {case['provider']}/{case['model']}")
    
    if result.errors:
        print(f"Error: {result.errors}")
    else:
        print(f"Energy: {result.energy.value:.8f} kWh")
        print(f"GWP: {result.gwp.value:.8f} kgCO2eq")
        print(f"ADPe: {result.adpe.value:.8f} kgSbeq")
        print(f"PE: {result.pe.value:.8f} MJ")
        print(f"WCF: {result.wcf.value:.8f} kgH2Oeq")
        print(f"Usage GWP: {result.usage.gwp.value:.8f} kgCO2eq")
        print(f"Embodied GWP: {result.embodied.gwp.value:.8f} kgCO2eq")
        
    
       
    



The model architecture has not been released, expect lower precision. For further information visit https://ecologits.ai/tutorial/warnings_and_errors/#model-arch-not-released
The model architecture is multimodal, expect lower precision. For further information visit https://ecologits.ai/tutorial/warnings_and_errors/#model-arch-multimodal


LLM Impacts test

Model name: openai/gpt-4o-mini
Energy: 0.00003510 [0.00003237 - 0.00003783] kWh
GWP: 0.00001553 [0.00001448 - 0.00001658] kgCO2eq
ADPe: 0.00000000 [0.00000000 - 0.00000000] kgSbeq
PE: 0.00036499 [0.00033851 - 0.00039147] MJ
WCF: 0.00077198 [0.00076015 - 0.00078381] kgH2Oeq
Usage GWP: 0.00001346 [0.00001241 - 0.00001451] kgCO2eq
Embodied GWP: 0.00000207 [0.00000207 - 0.00000207] kgCO2eq

Model name: anthropic/claude-haiku-4-5
Energy: 0.00004559 [0.00002529 - 0.00006589] kWh
GWP: 0.00001912 [0.00001079 - 0.00002745] kgCO2eq
ADPe: 0.00000000 [0.00000000 - 0.00000000] kgSbeq
PE: 0.00046141 [0.00025819 - 0.00066464] MJ
WCF: 0.00068595 [0.00041670 - 0.00095520] kgH2Oeq
Usage GWP: 0.00001749 [0.00000970 - 0.00002527] kgCO2eq
Embodied GWP: 0.00000164 [0.00000109 - 0.00000218] kgCO2eq

Model name: openai/gpt-4o
Energy: 0.00084483 [0.00065241 - 0.00103724] kWh
GWP: 0.00035272 [0.00027892 - 0.00042653] kgCO2eq
ADPe: 0.00000000 [0.00000000 - 0.00000000] kgSbeq
PE: 0.00853073 [0.0

In [7]:
# test training

test_cases = [
    {'provider': 'openai', 'model': 'gpt-4o-mini', 'tokens': 400},
]


for case in test_cases:
    result = llm_train_impacts(
        provider=case['provider'],
        model_name=case['model'],
        output_token_count=case['tokens'],
        electricity_mix_zone='USA'
    )
    
    print(f"\nModel name: {case['provider']}/{case['model']}")
    
    if result.errors:
        print(f"Error: {result.errors}")
    else:
        print(f"Energy: {result.energy.value:.8f} kWh")
        print(f"GWP: {result.gwp.value:.8f} kgCO2eq")
        print(f"ADPe: {result.adpe.value:.8f} kgSbeq")
        print(f"PE: {result.pe.value:.8f} MJ")
        print(f"WCF: {result.wcf.value:.8f} kgH2Oeq")
        print(f"Usage GWP: {result.usage.gwp.value:.8f} kgCO2eq")
        print(f"Embodied GWP: {result.embodied.gwp.value:.8f} kgCO2eq")
        
    

min=8.037020945190754e-11 max=5.539925908651576e-10

Model name: openai/gpt-4o-mini
Energy: 0.00000000 [0.00000000 - 0.00000000] kWh
GWP: 0.00000000 [0.00000000 - 0.00000000] kgCO2eq
ADPe: 0.00000000 [0.00000000 - 0.00000000] kgSbeq
PE: 0.00000000 [0.00000000 - 0.00000001] MJ
WCF: 0.00000000 [0.00000000 - 0.00000001] kgH2Oeq
Usage GWP: 0.00000000 [0.00000000 - 0.00000000] kgCO2eq
Embodied GWP: 0.00000000 [0.00000000 - 0.00000000] kgCO2eq


In [8]:
# test training - data storage

test_cases = [
    {'provider': 'openai', 'model': 'gpt-4o-mini', 'tokens': 400},
]


for case in test_cases:
    result = llm_train_data_storage_impacts(
        provider=case['provider'],
        model_name=case['model'],
        output_token_count=case['tokens'],
        electricity_mix_zone='USA'
    )
    
    print(f"\nModel name: {case['provider']}/{case['model']}")
    
    if result.errors:
        print(f"Error: {result.errors}")
    else:
        print(f"Energy: {result.energy.value:.8f} kWh")
        print(f"GWP: {result.gwp.value:.8f} kgCO2eq")
        print(f"ADPe: {result.adpe.value:.8f} kgSbeq")
        print(f"PE: {result.pe.value:.8f} MJ")
        print(f"WCF: {result.wcf.value:.8f} kgH2Oeq")
        print(f"Usage GWP: {result.usage.gwp.value:.8f} kgCO2eq")
        print(f"Embodied GWP: {result.embodied.gwp.value:.8f} kgCO2eq")

min=5.8346796232079735e-15 max=1.1491000044393076e-14

Model name: openai/gpt-4o-mini
Energy: 0.00000000 [0.00000000 - 0.00000000] kWh
GWP: 0.00000000 [0.00000000 - 0.00000000] kgCO2eq
ADPe: 0.00000000 [0.00000000 - 0.00000000] kgSbeq
PE: 0.00000000 [0.00000000 - 0.00000000] MJ
WCF: 0.00000000 [0.00000000 - 0.00000000] kgH2Oeq
Usage GWP: 0.00000000 [0.00000000 - 0.00000000] kgCO2eq
Embodied GWP: 0.00000000 [0.00000000 - 0.00000000] kgCO2eq
